In [1]:
!pip install transformers accelerate bitsandbytes
!pip install flask pyngrok -q
!pip install torchao --upgrade -q
!pip install peft==0.10.0 huggingface_hub==0.23.0 -q

In [2]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

print("✅ Token loaded!")

✅ Token loaded!


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download
import safetensors.torch
import torch

base_model_name = "mistralai/Mistral-7B-Instruct-v0.2"

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    token=hf_token
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name, token=hf_token)



adapter_path = hf_hub_download(
    repo_id="GRMenon/mental-health-mistral-7b-instructv0.2-finetuned-V2",
    filename="adapter_model.safetensors",
    token=hf_token
)

adapter_weights = safetensors.torch.load_file(adapter_path)
model.load_state_dict(adapter_weights, strict=False)



Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

_IncompatibleKeys(missing_keys=['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.v_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.1.self_attn.k_proj.weight', 'model.layers.1.self_attn.v_proj.weight', 'model.layers.1.self_attn.o_proj.weight', 'model.layers.1.mlp.gate_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.1.post_attention_layernorm.weight', 'model.layers.2.self_attn.q_proj.weight', 'model.layers.2.self_attn.k_proj.weight', 'model.layers.2.self_attn.v_proj.weight', 'model.layers.2.self_attn.o_proj.weight', 'model.layers.2.mlp.gat

In [4]:
def chat(message, history=[]):
    system = """You are a warm, empathetic mental health counselor working with college students aged 18-25.
Rules:
- Ask only ONE question at a time
- Be gentle and conversational
- Keep responses short (2-3 sentences max)
- Explore feelings before jumping to solutions
- Never give a list of questions"""

    # Build conversation history
    conv = f"<s>[INST] <<SYS>>\n{system}\n<</SYS>>\n\n"
    
    for user_msg, bot_msg in history:
        conv += f"{user_msg} [/INST] {bot_msg} </s><s>[INST] "
    
    conv += f"{message} [/INST]"
    
    inputs = tokenizer(conv, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    new_tokens = outputs[0][inputs.input_ids.shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

# Test conversation
history = []

msg1 = "I feel lost and don't know what career to choose"
reply1 = chat(msg1, history)
print(f"Student: {msg1}")
print(f"Counselor: {reply1}\n")
history.append((msg1, reply1))

msg2 = "I like technology but my parents want me to be a doctor"
reply2 = chat(msg2, history)
print(f"Student: {msg2}")
print(f"Counselor: {reply2}")

Student: I feel lost and don't know what career to choose
Counselor: I'm here to listen. It can be overwhelming to decide on a career path. Have you considered what subjects or activities bring you the most joy and fulfillment? Exploring those areas might lead you to a career that aligns with your passions.

Student: I like technology but my parents want me to be a doctor
Counselor: I understand that it can be challenging to balance your interests and your parents' expectations. It's important to remember that ultimately, you'll be the one spending the most time in your career. Have you considered exploring ways to combine your interest in technology with a career in healthcare, such as becoming a medical technologist or a health informatics specialist?


In [5]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading
import torch
import gc

# (Assuming your model and tokenizer initialization code is right above this)

app = Flask(__name__)

@app.route('/chat', methods=['POST'])
def chat_api():
    data = request.json
    prompt = data.get('prompt', '')
    
    # Grab parameters with default fallbacks
    max_tokens = data.get('max_new_tokens', 300) 
    temperature = data.get('temperature', 0.7)
    repetition_penalty = data.get('repetition_penalty', 1.15) # Added to stop repetitive tics
    
    torch.cuda.empty_cache()
    gc.collect()
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096 # Increased from 1024 to stop context window truncation
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            repetition_penalty=repetition_penalty, # Applied here
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    new_tokens = outputs[0][inputs.input_ids.shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    del inputs, outputs
    torch.cuda.empty_cache()
    
    return jsonify({'response': response.strip()})

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok'})

# Put your actual token back in here
ngrok.set_auth_token("<token>g")
public_url = ngrok.connect(5000)
print(f"🔥 API URL: {public_url}")

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()

🔥 API URL: NgrokTunnel: "https://9df5-34-132-209-50.ngrok-free.app" -> "http://localhost:5000"


In [ ]:
#2v<>zvuH_Lg